# 02 — Preprocessing

Clean the raw PJME series and write train/test parquet files for the SARIMA notebook.

Inputs: `data/raw/PJME_hourly.csv`  
Outputs: `data/processed/train.parquet`, `data/processed/test.parquet`

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RAW_CSV = PROJECT_ROOT / 'data' / 'raw' / 'PJME_hourly.csv'
PROCESSED = PROJECT_ROOT / 'data' / 'processed'
PROCESSED.mkdir(parents=True, exist_ok=True)

from src.data_loader import load_pjme
from src.preprocessing import ensure_hourly, train_test_split_timeseries

In [2]:
raw = load_pjme(RAW_CSV)
clean = ensure_hourly(raw)
print(f'Imputed rows: {clean["imputed"].sum()} of {len(clean):,}')

Imputed rows: 30 of 145,392


In [3]:
train, test = train_test_split_timeseries(clean[['PJME_MW']], test_hours=720, train_years=2)
print(f'Train: {len(train):,} rows  ({train.index.min()} -> {train.index.max()})')
print(f'Test:  {len(test):,} rows  ({test.index.min()} -> {test.index.max()})')

Train: 17,520 rows  (2016-07-04 01:00:00 -> 2018-07-04 00:00:00)
Test:  720 rows  (2018-07-04 01:00:00 -> 2018-08-03 00:00:00)


In [4]:
train.to_parquet(PROCESSED / 'train.parquet')
test.to_parquet(PROCESSED / 'test.parquet')
print('Wrote train.parquet and test.parquet to', PROCESSED)

Wrote train.parquet and test.parquet to /Users/karrar/Documents/GitHub/BA26-01-Time-Series/Code/Python/EnergyDemandForecaster/data/processed
